# Load Packages

In [1]:
import torch
from transformers import pipeline
import pandas as pd
import numpy as np
import os

c:\Users\bilso\anaconda3\envs\bert-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Parameters

In [2]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "Outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_dir = os.path.join("..", "Data")

# Seed for reproducibility
np.random.seed(42)

# Load and Pre-Process Data

In [3]:
# Load dataset
toaster_file_path = os.path.join(data_dir, "Param_Toaster Data_2018_23.xlsx")
toaster_df = pd.read_excel(toaster_file_path)

# Data preview
toaster_df.head()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,RV_DT,VP,HLP_VT,IMG_PRST,TTL_RV,RVS_L,RV_TRANS,SUBJ,SRVS,CP_RVS
0,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
1,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
2,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
3,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/B...,...,2018-01-01,1,12,1,863,413,"I bought this for me husband for Christmas, af...",0.508333,positive,0.4754
4,B07H81RZ9Q,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,58.890000,0.000000,58.89,1,0,4.2,9529,https://www.amazon.com/stores/HamiltonBeach/pa...,...,2018-01-01,1,1,0,4979,106,"Worked OK, never above average. Died one year ...",0.45,negative,-0.6908


In [4]:
# Size before deduplication
print(f"Dataset size before deduplication: {toaster_df.shape[0]} rows")

# Format RV_DT as datetime
toaster_df["RV_DT"] = pd.to_datetime(toaster_df["RV_DT"])

# Remove duplicate rows by keeping the first date (RV_DT) for each unique combination of reviewer (RVR) and product (ASIN)
toaster_df = (
    toaster_df.sort_values(["RVR", "ASIN", "RV_DT"], ascending=[True, True, False])
    .drop_duplicates(subset=["RVR", "ASIN"], keep="first")
    .reset_index(drop=True)
)

# Size after deduplication
print(f"Dataset size after deduplication: {toaster_df.shape[0]} rows")

Dataset size before deduplication: 85262 rows
Dataset size after deduplication: 62014 rows


# Sentiment Analysis

In [5]:
print(toaster_df[["RV_TRANS", "RV_DT"]].head(10))

                                            RV_TRANS      RV_DT
0                    Love making 4 slices at a time. 2019-01-06
1  Great item to use if you love grilled cheese b... 2018-12-09
2  I had high hopes for this toaster  - but it ta... 2021-03-28
3  BE AWARE: Read all the instructions included a... 2022-11-17
4                                                  . 2020-10-04
5  This oven works very well, and does everything... 2022-10-03
6  I received this toaster as a gift. Really grea... 2021-01-25
7  Love this toaster, it has super wide slots tha... 2020-05-18
8  Clear view to toasting indeed. Have not had is... 2021-03-23
9  Only toasts one side of the sandwich. Slots ar... 2021-01-15


In [6]:
#model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
#model_name = "distilbert-base-uncased-finetuned-sst-2-english"
#model_name = "cardiffnlp/twitter-roberta-base-sentiment"
model_name = "microsoft/deberta-v3-base"
classifier = pipeline("sentiment-analysis", model=model_name)

result = classifier("I love this product!")
print(result)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 23330.49it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING   

[{'label': 'LABEL_1', 'score': 0.6115269660949707}]


In [ ]:
# Load sentiment analysis pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# perform sentiment analysis on RV_TRANS column using sliding window approach
def analyze_sentiment_long_text(text, max_length=512, overlap=50):
    """Analyze sentiment for long texts using sliding window"""
    text = str(text)
    
    # If text is short enough, process directly
    if len(text) <= max_length:
        return classifier(text)[0]
    
    # Split into overlapping chunks
    chunks = []
    for i in range(0, len(text), max_length - overlap):
        chunk = text[i:i + max_length]
        chunks.append(chunk)
    
    # Analyze each chunk and aggregate results
    sentiments = []
    scores = []
    for chunk in chunks:
        result = classifier(chunk)[0]
        sentiments.append(result["label"])
        scores.append(result["score"])
    
    # Majority vote for sentiment, average for score
    from collections import Counter
    most_common_sentiment = Counter(sentiments).most_common(1)[0][0]
    avg_score = np.mean(scores)
    
    return {"label": most_common_sentiment, "score": avg_score}

# Apply to all rows
toaster_df["RV_TRANS_SENTIMENT"] = toaster_df["RV_TRANS"].apply(analyze_sentiment_long_text)

# Print the results
print(toaster_df[["RV_TRANS", "RV_TRANS_SENTIMENT"]].head())